In [1]:
import matplotlib.pyplot as plt
import sys
from typing import List, Dict
if not '../..' in sys.path:
    sys.path.append('../..')
import basico
import numpy
import pandas
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import random
import itertools
import math
import pickle
import time
import os

In [ ]:
fname = "../Chass_simulation_data.xlsx"
data = pandas.read_excel(fname, sheet_name=1)

data

,time,adp,asa,asp,aspp,atp,hs,hsp,nadp,nadph,phos,thr,Experiment
0,0,0.000000,0.000000,1.292900,0.000000,6.464400,0.000000,0.000000,0.000000,2.000000,0.000000,2.000000,1
1,1,0.000000,0.013994,1.123084,0.003921,4.333367,0.000000,0.018433,0.000000,1.836330,0.000000,1.747953,1
2,2,0.000000,0.021633,1.984576,0.004058,9.364119,0.004227,0.013632,0.000000,1.878701,0.000000,1.758829,1
3,3,0.013249,0.027716,0.592398,0.003463,6.618906,0.000000,0.006901,0.484441,1.879578,0.000000,1.947638,1
4,4,0.000000,0.046001,1.100811,0.004420,5.978385,0.076172,0.033020,0.031116,1.765170,0.415744,1.651898,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
904,96,2.263438,0.009160,1.140171,0.003236,5.644891,0.026977,0.024238,1.472780,0.436031,2.321581,3.015630,9
905,97,2.310092,0.004245,0.999620,0.001421,10.916923,0.018037,0.000000,1.613465,0.092504,2.098442,2.570191,9
906,98,2.509200,0.003017,1.258741,0.002973,7.784833,0.002394,0.028848,1.855979,0.022572,2.544461,2.774298,9
907,99,1.860823,0.007597,1.402961,0.002020,8.551870,0.002968,0.011389,1.798756,0.204134,2.269433,2.993754,9


In [3]:
with open("../chass_testbatch.txt", "r") as f:
    testbatch_str = f.read().split()
    testbatch = set([int(el) for el in testbatch_str])
prev_val = 0
batches_lengths = []
for val in data["Experiment"]:
    if val != prev_val:
        batches_lengths.append(1)
        prev_val = val
    else:
        batches_lengths[-1] += 1
not_testbatch = set(range(len(batches_lengths))) - testbatch
not_testbatch_exp_labels = [el + 1 for el in not_testbatch]

In [4]:
maxes = {}
mins = {}
concentrations_labels = ['adp', 'asa', 'asp', 'aspp', 'atp', 'hs', 'hsp', 'nadp', 'nadph', 'phos', 'thr']
for label in (['time'] + concentrations_labels):
    max_val = max(data[data["Experiment"].isin(not_testbatch_exp_labels)][label])
    min_val = min(data[data["Experiment"].isin(not_testbatch_exp_labels)][label])
    label_scaled = f"{label}_scaled"
    maxes[label] = max_val
    mins[label] = min_val
    data[label_scaled] = (data[label] - min_val) / (max_val - min_val)

In [5]:
data

,time,adp,asa,asp,aspp,atp,hs,hsp,nadp,nadph,...,asa_scaled,asp_scaled,aspp_scaled,atp_scaled,hs_scaled,hsp_scaled,nadp_scaled,nadph_scaled,phos_scaled,thr_scaled
0,0,0.000000,0.000000,1.292900,0.000000,6.464400,0.000000,0.000000,0.000000,2.000000,...,0.000000,0.394800,0.000000,0.353256,0.000000,0.000000,0.000000,0.829426,0.000000,0.252025
1,1,0.000000,0.013994,1.123084,0.003921,4.333367,0.000000,0.018433,0.000000,1.836330,...,0.207286,0.342945,0.494993,0.236803,0.000000,0.110997,0.000000,0.761550,0.000000,0.140455
2,2,0.000000,0.021633,1.984576,0.004058,9.364119,0.004227,0.013632,0.000000,1.878701,...,0.320438,0.606010,0.512326,0.511716,0.012322,0.082088,0.000000,0.779122,0.000000,0.145269
3,3,0.013249,0.027716,0.592398,0.003463,6.618906,0.000000,0.006901,0.484441,1.879578,...,0.410557,0.180895,0.437225,0.361700,0.000000,0.041554,0.220637,0.779485,0.000000,0.228847
4,4,0.000000,0.046001,1.100811,0.004420,5.978385,0.076172,0.033020,0.031116,1.765170,...,0.681407,0.336144,0.557991,0.326697,0.222028,0.198841,0.014172,0.732039,0.141222,0.097935
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
904,96,2.263438,0.009160,1.140171,0.003236,5.644891,0.026977,0.024238,1.472780,0.436031,...,0.135682,0.348163,0.408606,0.308473,0.078633,0.145956,0.670771,0.180828,0.788609,0.701602
905,97,2.310092,0.004245,0.999620,0.001421,10.916923,0.018037,0.000000,1.613465,0.092504,...,0.062876,0.305244,0.179426,0.596571,0.052574,0.000000,0.734846,0.038363,0.712812,0.504425
906,98,2.509200,0.003017,1.258741,0.002973,7.784833,0.002394,0.028848,1.855979,0.022572,...,0.044688,0.384369,0.375379,0.425413,0.006977,0.173715,0.845298,0.009361,0.864319,0.594774
907,99,1.860823,0.007597,1.402961,0.002020,8.551870,0.002968,0.011389,1.798756,0.204134,...,0.112527,0.428408,0.255025,0.467329,0.008651,0.068584,0.819236,0.084657,0.770896,0.691918


In [6]:
batches = []
for i, length in enumerate(batches_lengths):
    batches.append(data.iloc[i*length : i*length + length].reset_index(drop=True))
length = batches_lengths[0]

#scale inputs
changing_vars = ['asp', 'atp']
first_of_each_batch = data[changing_vars].iloc[::length]
mins_Xs = first_of_each_batch.min()
maxes_Xs = first_of_each_batch.max()
scaled_initial_conditions = (first_of_each_batch - mins_Xs)/(maxes_Xs-mins_Xs)

Xs_ys = []
for i, batch in enumerate(batches):
    scaled_initials_for_this_batch = pandas.concat([scaled_initial_conditions.iloc[i:i+1]]*length, ignore_index=True)
    X = pandas.concat([batch[["time_scaled"]], scaled_initials_for_this_batch], axis=1)
    y = batch[[f"{label}_scaled" for label in concentrations_labels]]
    X = torch.Tensor(X.values)
    y = torch.Tensor(y.values)
    Xs_ys.append((X, y))

In [ ]:
INPUTS = 3
OUTPUTS = 11
NUM_HIDDEN = 1
DROPOUT = 0
STOCH_REG_BATCH_SIZE = 2
LR = 0.0001

class ChassRnnreluRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(ChassRnnreluRegressor, self).__init__()
        
        # Number of hidden dimensions
        self.hidden_dim = hidden_dim
        
        # Number of hidden layers
        self.layer_dim = layer_dim
        
        # RNN
        self.rnn = nn.RNN(input_dim, hidden_dim, layer_dim, batch_first=True, dropout=DROPOUT, nonlinearity='relu')
        
        # Readout layer
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        
        # Initialize hidden state with zeros
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(device='cuda')
            
        # One time step
        out, hn = self.rnn(x, h0)
        out = self.fc(out) 
        return out

loss_fn = nn.MSELoss()

In [8]:
n_epochs = 1000000

num_batches = len(batches_lengths)
with open("../chass_train_validation_test", "rb") as file:
    train_validation_test = pickle.load(file)
nums_nodes_one_layer = [10, 15, 18, 21]
num_repetitions = 3
time_to_wait_for_improvement = 2000

In [9]:
num_train_batches = len(train_validation_test[0][0])
example_y = Xs_ys[0][1]
n_train = example_y.numel() * num_train_batches
print(f"Number of data points is {n_train}")

Number of data points is 5555


In [10]:
def AICc(nw, model, n_train, dataloader):
    for X, y in dataloader:
        y_pred = model(X)
        Train_Loss = loss_fn(y_pred, y)
        aic = n_train * math.log(Train_Loss) + 2*nw + 2*nw*(nw+1)/(n_train-nw-1)
        return aic

In [11]:
counter = 0
while os.path.exists(f"chass_rnn_relu_{counter}.csv"):
    counter += 1
csv_name = f"chass_rnn_relu_{counter}.csv"

In [12]:
newpath = "saved_models"
if not os.path.exists(newpath):
    os.makedirs(newpath)

In [ ]:
csv_counter = 0
csv_max = 100
csv_first = True
columns = ['num_nodes_on_layer', 'start_weights_iteration', 'k_fold', 'epoch', 'train', 'validation']
nums_nodes_on_layer_for_csv = []
start_weights_iteration = []
k_folds = []
epochs = []
train_losses = []
validation_losses = []
for num_nodes in nums_nodes_one_layer:
    best_validation_loss_model_global = None
    best_validation_loss_global = math.inf
    best_train_val_test_global = None
    for j in range(num_repetitions):
        nn_model = ChassRnnreluRegressor(INPUTS, num_nodes, NUM_HIDDEN, OUTPUTS)
        number_of_parameters = sum(torch.numel(tensor) for tensor in nn_model.state_dict().values())
        models = []
        test_losses = []
        train_losses_end = []
        val_losses_end = []
        nums_epoch = []
        AICs = []
        ti = time.time()
        for i, (train_batches_nums, validation_batches_nums, test_batches_nums) in enumerate(train_validation_test):


            best_validation_loss_model = None
            best_validation_loss = math.inf
            
            train_batches = [Xs_ys[i] for i in train_batches_nums]
            validation_batches = [Xs_ys[i] for i in validation_batches_nums]
            test_batches = [Xs_ys[i] for i in test_batches_nums]
            
            X_train = torch.stack([batch[0] for batch in train_batches])
            y_train = torch.stack([batch[1] for batch in train_batches])
            X_validation = torch.stack([batch[0] for batch in validation_batches])
            y_validation = torch.stack([batch[1] for batch in validation_batches])
            X_test = torch.stack([batch[0] for batch in test_batches])
            y_test = torch.stack([batch[1] for batch in test_batches])
            
            X_train = X_train.to(device='cuda')
            y_train = y_train.to(device='cuda')
            X_validation = X_validation.to(device='cuda')
            y_validation = y_validation.to(device='cuda')
            X_test = X_test.to(device='cuda')
            y_test = y_test.to(device='cuda')
            
            training_data = TensorDataset(X_train, y_train)
            validation_data = TensorDataset(X_validation, y_validation)
            test_data = TensorDataset(X_test, y_test)
            
            train_dataloader = DataLoader(training_data, batch_size=STOCH_REG_BATCH_SIZE, shuffle=True)
            val_dataloader = DataLoader(validation_data, batch_size=len(validation_data))
            test_dataloader = DataLoader(test_data, batch_size=len(test_data))
            this_iteration_model = ChassRnnreluRegressor(INPUTS, num_nodes, NUM_HIDDEN, OUTPUTS)
            this_iteration_model.load_state_dict(nn_model.state_dict())
            this_iteration_model.to(device='cuda')
            optimizer = optim.Adam(this_iteration_model.parameters(), lr=LR, )
            counter = time_to_wait_for_improvement
            
            for epoch in range(n_epochs):
                this_iteration_model.train()
                for X, y in train_dataloader:
                    y_pred = this_iteration_model(X)
                    loss = loss_fn(y_pred, y)
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                #print(f'Finished epoch {epoch}, latest loss {loss}')
                train_losses.append(float(loss))
                sum_loss = 0
                num_loss = 0
                with torch.no_grad():
                    this_iteration_model.eval()
                    for X, y in val_dataloader:
                        y_pred = this_iteration_model(X)
                        loss = loss_fn(y_pred, y)
                        sum_loss += loss
                        num_loss += 1
                    mean_loss = float(sum_loss / num_loss)
                if mean_loss < best_validation_loss: #- minimal_improvement:
                    counter = time_to_wait_for_improvement
                    best_validation_loss = mean_loss
                    best_validation_loss_model = ChassRnnreluRegressor(INPUTS, num_nodes, NUM_HIDDEN, OUTPUTS)
                    best_validation_loss_model.to(device='cuda')
                    best_validation_loss_model.load_state_dict(this_iteration_model.state_dict())
                counter -= 1
                if counter < 0:
                    nums_epoch.append(epoch)
                    break
                validation_losses.append(mean_loss)
                #print("validation loss", mean_loss)
                nums_nodes_on_layer_for_csv.append(str(num_nodes))
                start_weights_iteration.append(j)
                k_folds.append(i)
                epochs.append(epoch)
                csv_counter += 1
                if csv_counter == 100:
                    csv_counter = 0
                    df = pandas.DataFrame(zip(nums_nodes_on_layer_for_csv,
                                              start_weights_iteration,
                                              k_folds,
                                              epochs,
                                              train_losses,
                                              validation_losses), columns = columns)
                    if csv_first:
                        csv_first = False
                        df.to_csv(csv_name, index=False)
                    else:
                        df.to_csv(csv_name, mode='a', index=False, header=False)
                    nums_nodes_on_layer_for_csv = []
                    start_weights_iteration = []
                    k_folds = []
                    epochs = []
                    train_losses = []
                    validation_losses = []
            else:
                nums_epoch.append(n_epochs)

            sum_loss = 0
            num_loss = 0
            
            elapsed_time = time.time() - ti
            torch.save(best_validation_loss_model.state_dict(), f"saved_models/Chass_rnn_relu_{num_nodes}_{j}_{i}")
            
            if best_validation_loss < best_validation_loss_global:
                best_validation_loss_global = best_validation_loss
                best_validation_loss_model_global = best_validation_loss_model
                best_train_val_test_global = [train_batches_nums, validation_batches_nums, test_batches_nums]

            sum_loss = 0
            num_loss = 0
            with torch.no_grad():
                best_validation_loss_model.eval()
                for X, y in test_dataloader:
                    y_pred = best_validation_loss_model(X)
                    loss = loss_fn(y_pred, y)
                    sum_loss += loss
                    num_loss += 1
                mean_test_loss = float(sum_loss / num_loss)

                test_losses.append(mean_test_loss)

                total_train_dataloader = DataLoader(training_data, len(training_data))
                sum_loss = 0
                num_loss = 0
                for X, y in total_train_dataloader:
                    y_pred = best_validation_loss_model(X)
                    Train_Loss = loss_fn(y_pred, y)
                    sum_loss += Train_Loss
                    num_loss += 1
                train_losses_end.append(float(sum_loss/num_loss))
                total_val_dataloader = DataLoader(validation_data, len(validation_data))
                sum_loss = 0
                num_loss = 0
                for X, y in total_val_dataloader:
                    y_pred = best_validation_loss_model(X)
                    Val_Loss = loss_fn(y_pred, y)
                    sum_loss += Val_Loss
                    num_loss += 1
                val_losses_end.append(float(sum_loss/num_loss))
                AICs.append(AICc(number_of_parameters, best_validation_loss_model, n_train, total_train_dataloader))
                

        elapsed_time = time.time() - ti
        mean_test = numpy.mean(test_losses)
        sd_test = numpy.std(test_losses, ddof=1)
        mean_AIC = numpy.mean(AICs)
        sd_AIC = numpy.std(AICs, ddof=1)
        print(mean_test)
        print(sd_test)
        print(nums_epoch)
        with open("Chass_rnn_relu.txt", "a") as f:
            f.write(f"dropout {DROPOUT}, number of nodes on layers {num_nodes} mean test loss {mean_test}, sd {sd_test}, number of parameters {number_of_parameters}, batch size {STOCH_REG_BATCH_SIZE}, learning rate {LR}\n")
            f.write(f"{test_losses}\n")
            f.write(f"mean AICc: {mean_AIC}, sd AICc: {sd_AIC}\n")
            f.write(f"AICcs: {AICs}\n")
            f.write(f'time {elapsed_time}\n')
            f.write(f"iterations {nums_epoch}\n")
            f.write(f"validation losses {val_losses_end}\n")
            f.write(f"train losses {train_losses_end}\n\n")
    torch.save(best_validation_loss_model_global.state_dict(), f"saved_models/Chass_rnn_relu_{num_nodes}_best")
    with open(f'saved_models/Chass_rnn_relu_{num_nodes}_best_train_validation_test.pkl', 'wb') as f:
        pickle.dump(best_train_val_test_global, f)
df = pandas.DataFrame(zip(nums_nodes_on_layer_for_csv,
                          start_weights_iteration,
                          k_folds,
                          epochs,
                          train_losses,
                          validation_losses), columns = columns)
if csv_first:
    csv_first = False
    df.to_csv(csv_name, index=False)
else:
    df.to_csv(csv_name, mode='a', index=False, header=False)

0.009238699544221162
0.0005185349147320774
[11662, 14331, 10524, 45635, 32486, 12385, 8006, 22740, 16315, 23178]
0.009248949214816093
0.0004246274602975027
[14025, 47594, 12229, 41557, 14164, 14906, 13752, 21494, 22339, 24541]
0.010606824234127999
0.0027113717740479015
[45412, 5063, 15499, 11661, 8580, 44709, 12065, 32872, 23604, 35742]
0.00913770142942667
0.0005207492718195318
[17838, 15143, 13991, 17184, 18506, 21411, 18849, 8382, 20515, 21675]
0.009196785371750593
0.00045073082612682075
[7825, 14863, 12298, 14952, 11021, 11056, 14515, 13729, 12574, 15091]
0.009096930269151926
0.0005293013415150441
[8212, 22143, 7849, 51326, 10385, 7913, 14868, 29697, 13659, 32005]
0.008982924465090036
0.00045764481645575744
[17261, 7289, 9177, 21198, 8875, 11976, 6456, 21149, 23029, 22503]
0.008706704434007406
0.0002069612332768759
[26347, 35599, 16381, 21236, 9036, 24595, 34851, 7467, 11129, 14206]
0.008768274262547494
0.0001361785371700422
[12079, 8077, 7436, 10514, 7051, 12304, 10843, 9716, 11346